In [1]:
# %% [markdown]
# # Extracción de embeddings + clustering de sentidos (HDBSCAN)
#
# Requiere: transformers, torch, umap-learn, hdbscan, pandas, numpy
# Ejecuta: pip install transformers torch umap-learn hdbscan pandas numpy --break-system-packages
#
# IMPORTANTE — supuesto que debes verificar antes de correr todo el corpus:
# `id_vector` en tu CSV original asume que es la posición del sub-token DENTRO
# de la secuencia tokenizada, SIN contar los tokens especiales (CLS/BOS, SEP/EOS).
# Si tu extracción original sí los contaba, ajusta SPECIAL_TOKEN_OFFSET más abajo.
# Verifícalo con la celda de "sanity check" antes de confiar en resultados masivos.
 
# %%
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import umap
import hdbscan
from collections import defaultdict
import json
import matplotlib.pyplot as plt

C:\Users\msubr\Anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %% [markdown]
# ## Configuración — AJUSTAR por modelo
 
# %%
# Rellena con el identificador de Hugging Face o path local de cada modelo.
MODELOS = {
    "mex_state":   "guillermoruiz/mex_state", # BILMA variante Mexico (small/state)
    "mex_large":   "guillermoruiz/mex_large", # BILMA variante Mexico (large)
    "beto":        "dccuchile/bert-base-spanish-wwm-cased",                   
    "bertin":      "bertin-project/bertin-roberta-base-spanish",
    "robertuito":  "pysentimiento/robertuito-base-cased",
}

In [3]:
# Archivos generados en pasos anteriores
INTERSECCION_CSV = "output/interseccion_vocabulario.csv"
TOKENS_CSV = {  # CSV original completo (token, id_tweet, region, id_vector) por modelo
    "mex_state":   "output/tokens_mex_state.csv",
    "mex_large":   "output/tokens_mex_large.csv",
    "beto":        "output/tokens_beto.csv",
    "bertin":      "output/tokens_bertin.csv",
    "robertuito":  "output/tokens_robertuito.csv",
}
PALABRAS_CSV = {  # salida de reconstruir_palabras.py, por modelo
    "mex_state":   "output/words_mex_state.csv",
    "mex_large":   "output/words_mex_large.csv",
    "beto":        "output/words_beto.csv",
    "bertin":      "output/words_bertin.csv",
    "robertuito":  "output/words_robertuito.csv",
}
 
CAPA_EXTRACCION = -1          # penúltima capa oculta (mismo criterio para los 5 modelos)
SPECIAL_TOKEN_OFFSET = 0      # +1 si el tokenizador antepone CLS/BOS antes del primer id_vector=0
BATCH_SIZE = 32
MAX_LENGTH = 128              # los tweets son cortos; ajusta si truncas algo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
 
# Parámetros de HDBSCAN — mismos para los 5 modelos (fijar tras calibración)
UMAP_N_COMPONENTS = 12
HDBSCAN_MIN_CLUSTER_SIZE = 3
HDBSCAN_MIN_SAMPLES = 3
MIN_OCURRENCIAS_PARA_CLUSTER = 8   # palabras con menos ocurrencias no se agrupan

In [4]:
stopwords = ['algún','alguna','algunas','alguno','algunos','ambos','ampleamos','ante','antes','aquel','aquellas','aquellos','aqui','arriba','atras',
             'bajo','bastante','bien','cada','cierta','ciertas','cierto','ciertos','como','con','conseguimos','conseguir','consigo','consigue','consiguen',
             'consigues','cual','cuando','dentro','desde','donde','dos','el','ellas','ellos','empleais','emplean','emplear','empleas','empleo','en','encima',
             'entonces','entre','era','eramos','eran','eras','eres','es','esta','estaba','estado','estais','estamos','estan','estoy','fin','fue','fueron','fui',
             'fuimos','gueno','ha','hace','haceis','hacemos','hacen','hacer','haces','hago','incluso','intenta','intentais','intentamos','intentan',
             'intentar','intentas','intento','ir','la','largo','las','lo','los','mas','mientras','mio','modo','muchos','muy','nos','nosotros',
             'otro','para','pero','podeis','podemos','poder','podria','podriais','podriamos','podrian','podrias','por','por qué','porque','primero','puede','pueden',
             'puedo','quien','sabe','sabeis','sabemos','saben','saber','sabes','ser','si','siendo','sin','sobre','sois','solamente','solo','somos','soy','su',
             'sus','también','teneis','tenemos','tener','tengo','tiempo','tiene','tienen','todo','trabaja','trabajais','trabajamos','trabajan','trabajar','trabajas','trabajo',
             'tras','tuyo','ultimo','un','una','unas','uno','unos','usa','usais','usamos','usan','usar','usas','uso','va','vais','valor','vamos','van','vaya',
             'verdad','verdadera','verdadero','vosotras','vosotros','voy','yo','él','ésta','éstas','éste','éstos','última','últimas','último','últimos','a','añadió','aún',
             'actualmente','adelante','además','afirmó','agregó','ahí','ahora','al','algo','alrededor','anterior','apenas','aproximadamente','aquí','así','aseguró',
             'aunque','ayer','buen','buena','buenas','bueno','buenos','cómo','casi','cerca','cinco','comentó','conocer','consideró','considera','contra',
             'cosas','creo','cuales','cualquier','cuanto','cuatro','cuenta','da','dado','dan','dar','de','debe','deben','debido','decir','dejó','del','demás',
             'después','dice','dicen','dicho','dieron','diferente','diferentes','dijeron','dijo','dio','durante','e','ejemplo','ella','ello','embargo','encuentra',
             'esa','esas','ese','eso','esos','está','están','estaban','estar','estará','estas','este','esto','estos','estuvo','ex','existe','existen','explicó','expresó',
             'fuera','gran','grandes','había','habían','haber','habrá','hacerlo','hacia','haciendo','han','hasta','hay','haya','he','hecho','hemos','hicieron',
             'hizo','hoy','hubo','igual','indicó','informó','junto','lado','le','les','llegó','lleva','llevar','luego','lugar','más','manera','manifestó','mayor','me',
             'mediante','mejor','mencionó','menos','mi','misma','mismas','mismo','mismos','momento','mucha','muchas','mucho','nada','nadie','ni','ningún','ninguna',
             'ningunas','ninguno','ningunos','no','nosotras','nuestra','nuestras','nuestro','nuestros','nueva','nuevas','nuevo','nuevos','nunca','o','ocho',
             'otra','otras','otros','parece','parte','partir','pasada','pasado','pesar','poca','pocas','poco','pocos','podrá','podrán','podría','podrían','poner',
             'posible','próximo','próximos','primer','primera','primeros','principalmente','propia','propias','propio','propios','pudo','pueda','pues','qué','que',
             'quedó','queremos','quién','quienes','quiere','realizó','realizado','realizar','respecto','sí','sólo','se','señaló','sea','sean','según','segunda',
             'segundo','seis','será','serán','sería','sido','siempre','siete','sigue','siguiente','sino','sola','solas','solos','son','tal','tampoco',
             'tan','tanto','tenía','tendrá','tendrán','tenga','tenido','tercera','toda','todas','todavía','todos','total','trata','través','tres','tuvo',
             'usted','varias','varios','veces','ver','vez','y','ya', '_htag', '_usr', '_url', 'mi', 'tu', 'te', 'das', 'app', 'cuál', 'cuáles', 'cuándo', 'cuánto', 'puedes',
             'quienes', 'dónde', 'año', 'vs', 'día', 'pide', 'quot', 'años', 'Quiénes', 'video', 'Video', 'Acerca', 'with', 'that', 'by', 'to', 'for', 'and', 'of', 'the',
             'in', 'on', 'it', 'with', 'How', 'over', 'with', 'I', 'U', 's', 'from', 'after', 'his', 'is', 'as', 'will', 'at', 'an', 'new', 'are', 'has', 'who',
             'have', 'other', 'What', 'its', 'their', 'but', 'Why', 'not', 'more', 'we', 
             'aguascalientes', 'bc', 'bcs', 'campeche', 'chiapas', 'chihuahua', 'mexico_city', 
             'coahuila', 'colima', 'durango', 'guanajuato', 'guerrero', 'hidalgo', 'jalisco', 
             'michoacán', 'morelos', 'mexico', 'nayarit', 'nl', 'oaxaca', 'puebla', 'querétaro', 
             'qr', 'slp', 'sinaloa', 'sonora', 'tabasco', 'tamaulipas', 'tlaxcala', 
             'veracruz', 'yucatán', 'zacatecas']

def get_tweet(palabras_df, id_tweet, special_token=None):

    RED = "\033[91m"
    RESET = "\033[0m" 

   
    l = list(palabras_df[palabras_df['id_tweet']==id_tweet]['word_normalized'])
    l = [str(_) for _ in l]
    
    if special_token==None:
        print(" ".join(l))
        return
    
    s = ""
    for w in l:
        if w==special_token:
            s += RED + w + RESET + " "
        else:
            s += w + " "
    print(s)
    return 

In [5]:
# %% [markdown]
# ## 1. Cargar la lista de palabras objetivo (intersección de vocabulario)
 
# %%
interseccion = pd.read_csv(INTERSECCION_CSV)
interseccion = interseccion[~interseccion["word_normalized"].isin(stopwords)]
interseccion = interseccion[interseccion["word_normalized"].str.len()>3]
interseccion = interseccion[interseccion["word_normalized"].str.isalpha()]
palabras_objetivo = set(interseccion["word_normalized"])

print(f"Palabras objetivo: {len(palabras_objetivo)}")

Palabras objetivo: 12434


In [6]:
# %% [markdown]
# ## 2. Funciones de extracción
 
# %%
def construir_secuencias_por_tweet(tokens_path):
    """Agrupa el CSV completo de tokens en secuencias por id_tweet, ordenadas por id_vector."""
    df = pd.read_csv(tokens_path, keep_default_na=False)
    df = df.sort_values(["id_tweet", "id_vector"])
    secuencias = {}
    for id_tweet, grupo in df.groupby("id_tweet"):
        secuencias[id_tweet] = list(zip(grupo["id_vector"].tolist(), grupo["token"].tolist()))
    
        
    return secuencias
 
 
def extraer_embeddings_modelo(nombre_modelo, hf_id, tokens_path, palabras_path, palabras_objetivo):
    """
    Para un modelo: reconstruye secuencias de tokens por tweet, corre el modelo en
    lotes, y devuelve dos dicts alineados por índice:
      - embeddings_por_palabra: {palabra: [embeddings de cada ocurrencia]}
      - tweets_por_palabra: {palabra: [id_tweet de cada ocurrencia]}
    """
    print(f"\n--- {nombre_modelo} ---")
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModel.from_pretrained(hf_id, output_hidden_states=True).to(DEVICE).eval()
 
    secuencias = construir_secuencias_por_tweet(tokens_path)
 
    palabras_df = pd.read_csv(palabras_path)
    palabras_df = palabras_df[palabras_df["word_normalized"].isin(palabras_objetivo)]
    # ocurrencias por tweet, para saber qué extraer del forward pass de cada tweet
    ocurrencias_por_tweet = defaultdict(list)
    for _, row in palabras_df.iterrows():
        ocurrencias_por_tweet[row["id_tweet"]].append(
            (row["word_normalized"], 
             row["id_vector_start"], 
             row["id_vector_end"], 
             row['region']
            )
        )
 
    embeddings_por_palabra = defaultdict(list)
    tweets_por_palabra = defaultdict(list)
    ids_tweets = [t for t in ocurrencias_por_tweet.keys() if t in secuencias]
    print(f"Tweets a procesar: {len(ids_tweets)}")
 
    for i in range(0, len(ids_tweets), BATCH_SIZE):
        lote_ids = ids_tweets[i : i + BATCH_SIZE]
        lote_tokens = [[tok for _, tok in secuencias[tid]] for tid in lote_ids]
 
        encoded = tokenizer(
            lote_tokens,
            is_split_into_words=True,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
 
        with torch.no_grad():
            out = model(**encoded)
        hidden = out.hidden_states[CAPA_EXTRACCION]  # (batch, seq_len, hidden_dim)
 
        for b, tid in enumerate(lote_ids):
            
            for palabra, start, end, reg in ocurrencias_por_tweet[tid]:
                s = start + SPECIAL_TOKEN_OFFSET
                e = end + SPECIAL_TOKEN_OFFSET
                if e >= hidden.shape[1]:
                    continue  # se truncó por max_length, se descarta esta ocurrencia
                vec = hidden[b, s : e + 1, :].mean(dim=0).cpu().numpy()
                embeddings_por_palabra[palabra].append(vec)                
                
                tweets_por_palabra[palabra].append({'tweet_id': tid, 'region':reg})
 
        if i % (BATCH_SIZE * 50) == 0:
            print(f"  {i}/{len(ids_tweets)} tweets procesados")
 
    del model
    torch.cuda.empty_cache()
    return embeddings_por_palabra, tweets_por_palabra

In [34]:
# %% [markdown]
# ## 3. Sanity check — verificar el offset ANTES de correr todo
#
# Elige una palabra frecuente y un modelo, extrae 1-2 ocurrencias, e imprime
# el token en `id_vector_start` reconstruido para confirmar que corresponde
# realmente a la palabra esperada.
 
# %%
def sanity_check(nombre_modelo, hf_id, tokens_path, palabras_path, palabra_ejemplo):
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    secuencias = construir_secuencias_por_tweet(tokens_path)
    palabras_df = pd.read_csv(palabras_path)
    fila = palabras_df[palabras_df["word_normalized"] == palabra_ejemplo].iloc[0]
    tid, start, end = fila["id_tweet"], fila["id_vector_start"], fila["id_vector_end"]
    tokens_tweet = secuencias[tid]
    print("Tweet completo (tokens):", [t for _, t in tokens_tweet])
    print(f"Sub-tokens en id_vector [{start}, {end}] (sin offset):",
          [t for v, t in tokens_tweet if start <= v <= end])
 
# Ejemplo de uso — descomenta y ajusta:
# sanity_check("beto", MODELOS["beto"], TOKENS_CSV["beto"], PALABRAS_CSV["beto"], "banco")

In [35]:
sanity_check("beto", MODELOS["beto"], TOKENS_CSV["beto"], PALABRAS_CSV["beto"], "pobre")

Tweet completo (tokens): ['_', 'US', '##R', 'Pobre', 'vieja', 'ridícula', 'y', 'paté', '##tica', ',', 'ni', 'sabe', 'hablar', '.']
Sub-tokens en id_vector [4, 4] (sin offset): ['Pobre']


In [36]:
sanity_check("bertin", MODELOS["bertin"], TOKENS_CSV["bertin"], PALABRAS_CSV["bertin"], "pobre")

Tweet completo (tokens): ['_', 'US', 'R', 'ĠP', 'obre', 'Ġvieja', 'ĠridÃŃcula', 'Ġy', 'Ġpa', 'tÃ©tica', ',', 'Ġni', 'Ġsabe', 'Ġhablar', '.']
Sub-tokens en id_vector [4, 5] (sin offset): ['ĠP', 'obre']


In [37]:
sanity_check("robertuito", MODELOS["robertuito"], TOKENS_CSV["robertuito"], PALABRAS_CSV["robertuito"], "pobre")

Tweet completo (tokens): ['▁_', 'US', 'R', '▁Pobre', '▁vieja', '▁ridícula', '▁y', '▁paté', 'tica,', '▁ni', '▁sabe', '▁hablar.']
Sub-tokens en id_vector [4, 4] (sin offset): ['▁Pobre']


In [38]:
sanity_check("mex_state", MODELOS["mex_state"], TOKENS_CSV["mex_state"], PALABRAS_CSV["mex_state"], "pobre")

Tweet completo (tokens): ['Michoacán', '_GEO', '_USR', 'Pobre', 'vieja', 'rid', '##ícula', 'y', 'paté', '##tica', ',', 'ni', 'sabe', 'hablar', '.']
Sub-tokens en id_vector [4, 4] (sin offset): ['Pobre']


In [39]:
sanity_check("mex_large", MODELOS["mex_large"], TOKENS_CSV["mex_large"], PALABRAS_CSV["mex_large"], "pobre")

Tweet completo (tokens): ['Michoacán', '_GEO', '_USR', 'Pobre', 'vieja', 'rid', '##ícula', 'y', 'paté', '##tica', ',', 'ni', 'sabe', 'hablar', '.']
Sub-tokens en id_vector [4, 4] (sin offset): ['Pobre']


In [7]:
# %% [markdown]
# ## 4. Guardar embeddings (a nivel de palabra, antes de clusterizar)
#
# Guarda un .npz por modelo con dos arrays alineados por fila:
# `embeddings` (n_ocurrencias, dim) y `words` (n_ocurrencias,) — el
# word_normalized correspondiente a cada fila. Es la unidad de las 13,772
# palabras de la intersección (post mean-pooling), no sub-tokens individuales.
 
# %%
import os
 
def guardar_embeddings_modelo(nombre_modelo, embeddings_por_palabra, 
                              tweets_por_palabra,
                              carpeta_salida="embeddings"):
    os.makedirs(carpeta_salida, exist_ok=True)
    palabras_lista = []
    vectores = []
    for palabra, embs in embeddings_por_palabra.items():
        tpp = tweets_por_palabra[palabra]
        for v, d in zip(embs, tpp):
            palabras_lista.append({'palabra':palabra, 
                                   'tweet_id':d['tweet_id'],
                                   'region':d['region'] 
                                  })
            vectores.append(v)
    vectores = np.vstack(vectores)
    palabras_arr = np.array(palabras_lista)
    ruta = os.path.join(carpeta_salida, f"{nombre_modelo}_embeddings.npz")
    np.savez_compressed(ruta, embeddings=vectores, words=palabras_arr)
    print(f"  Guardado: {ruta} ({vectores.shape[0]} ocurrencias, dim={vectores.shape[1]})")
    return ruta

def cargar_embeddings(nombre_modelo, carpeta_salida='embeddings'):
    data = np.load(f"embeddings/{nombre_modelo}_embeddings.npz", allow_pickle=True)
    X, words = data["embeddings"], data["words"]
    return X, words

def cargar_tweets_por_palabra(nombre_modelo, carpeta_salida='embeddings'):
    with open(f"embeddings/{nombre_modelo}_tweets_por_palabra.json", 'r') as file:
        data = json.load(file)
    return data

In [8]:
# %% [markdown]
# ## 5. Clustering de sentidos por palabra (HDBSCAN)
#
# **`min_cluster_size` adaptativo, no fijo.** Un valor fijo (ej. 3) es razonable
# para una palabra con 20 ocurrencias, pero para una con 1,000+ ocurrencias es
# absurdamente permisivo — HDBSCAN encuentra decenas de micro-agrupaciones que
# son ruido estadístico, no sentidos reales (esto se confirmó con la corrida
# completa: 0% de palabras con k=1 en los 5 modelos, y k máximo de hasta 281 —
# imposible para polisemia real). La regla: min_cluster_size escala como una
# fracción de las ocurrencias de la palabra, con un piso y un techo.
 
# %%
MCS_FRACCION = 0.05   # min_cluster_size ≈ 5% de las ocurrencias de la palabra
MCS_MINIMO = 5
MCS_MAXIMO = 50
 
def calcular_min_cluster_size(n, fraccion=MCS_FRACCION, minimo=MCS_MINIMO, maximo=MCS_MAXIMO):
    return int(np.clip(round(fraccion * n), minimo, maximo))
 
 
def _hdbscan_labels(embeddings, min_cluster_size, min_samples):
    """Corre UMAP+HDBSCAN sobre los embeddings de una palabra y devuelve el
    array de labels (uno por ocurrencia, -1 = ruido)."""
    X = np.vstack(embeddings)
    n = len(embeddings)
    n_components = min(UMAP_N_COMPONENTS, n - 2)
    reducer = umap.UMAP(n_components=n_components)
    X_red = reducer.fit_transform(X)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples)
    return clusterer.fit_predict(X_red)
 
 
def clusterizar_palabra(embeddings, min_cluster_size=None, min_samples=None):
    """Dado un array de embeddings (n_ocurrencias, dim) para UNA palabra en UN modelo,
    devuelve (k_sentidos, pct_ruido, min_cluster_size_usado). Si no se pasa
    min_cluster_size explícito, se calcula de forma adaptativa según `n`
    (ver calcular_min_cluster_size) — NO se usa un valor fijo global."""
    n = len(embeddings)
    if n < MIN_OCURRENCIAS_PARA_CLUSTER:
        return None, None, None  # no evaluada, muy pocas ocurrencias
 
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(n)
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
 
    labels = _hdbscan_labels(embeddings, mcs, ms)
    k = len(set(labels) - {-1})
    pct_ruido = (labels == -1).mean() * 100
    return k, pct_ruido, mcs
 
 
def inspeccionar_ocurrencias(palabra, embeddings_dict, tweets_dict, min_cluster_size=None, min_samples=None):
    """Devuelve un DataFrame (id_tweet, cluster_sentido) para revisar caso por caso
    a qué tweet corresponde cada ocurrencia y en qué cluster (sentido) cayó."""
    embs = embeddings_dict[palabra]
    tids = tweets_dict[palabra]
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(len(embs))
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
    labels = _hdbscan_labels(embs, mcs, ms)
    return pd.DataFrame({"id_tweet": tids, "cluster_sentido": labels})

In [9]:
# %% [markdown]
# ## 6. Calibración de HDBSCAN — correr ANTES del pipeline completo
#
# Prueba varias combinaciones de `min_cluster_size`/`min_samples` sobre palabras
# de polisemia conocida en español. Revisa manualmente si el k_sentidos reportado
# es razonable (ej. "banco" debería salir con ≥2 sentidos, no 1 ni 6). Fija los
# valores de HDBSCAN_MIN_CLUSTER_SIZE / HDBSCAN_MIN_SAMPLES arriba en la
# configuración según lo que veas aquí, y no los vuelvas a tocar después de ver
# los resultados del corpus completo.
 
# %%
PALABRAS_CALIBRACION_POLISEMICAS = {"banco", "gato", "planta", "sierra", "cola", "papa", "vela", "muñeca"}
PALABRAS_CALIBRACION_MONOSEMICAS = {"bicicleta", "computadora", "cumpleaños", "desayuno",
                                      "aeropuerto", "estudiante", "diciembre", }
PALABRAS_CALIBRACION = PALABRAS_CALIBRACION_POLISEMICAS | PALABRAS_CALIBRACION_MONOSEMICAS

resultados_calibracion = []
embeddings_calib_por_modelo = {}   # se guarda por si quieres inspeccionar después
tweets_calib_por_modelo = {}
 
# min_cluster_size adaptativo (calcular_min_cluster_size) + un par de valores fijos
# de referencia, para comparar contra el enfoque anterior.
CONFIGS_CALIBRACION = ["adaptativo", 3, 5, 10]
 
for nombre_modelo, hf_id in MODELOS.items():
    embeddings_calib, tweets_calib = extraer_embeddings_modelo(
        nombre_modelo, hf_id, TOKENS_CSV[nombre_modelo], PALABRAS_CSV[nombre_modelo], PALABRAS_CALIBRACION
    )
    embeddings_calib_por_modelo[nombre_modelo] = embeddings_calib
    tweets_calib_por_modelo[nombre_modelo] = tweets_calib
    for palabra, embs in embeddings_calib.items():
        tipo_esperado = "polisémica" if palabra in PALABRAS_CALIBRACION_POLISEMICAS else "monosémica"
        for config in CONFIGS_CALIBRACION:
            mcs_explicito = None if config == "adaptativo" else config
            k, pct_ruido, mcs_usado = clusterizar_palabra(embs, min_cluster_size=mcs_explicito, min_samples=HDBSCAN_MIN_SAMPLES)
            resultados_calibracion.append({
                "modelo": nombre_modelo, "palabra": palabra, "tipo_esperado": tipo_esperado,
                "n_ocurrencias": len(embs), "config": config, "min_cluster_size_usado": mcs_usado,
                "k_sentidos": k, "pct_ruido": pct_ruido,
            })
 
calibracion_df = pd.DataFrame(resultados_calibracion)
print(calibracion_df.sort_values(["tipo_esperado", "palabra", "modelo", "config"], key=lambda s: s.astype(str)).to_string(index=False))
 
# Resumen rápido: ¿cuántas monosémicas dan k=1 con cada config?
print("\n% de palabras monosémicas de control con k_sentidos == 1, por config:")
print(calibracion_df[calibracion_df["tipo_esperado"] == "monosémica"]
      .groupby("config")["k_sentidos"].apply(lambda s: (s == 1).mean()))
 


--- mex_state ---


Loading weights: 100%|██████████| 101/101 [00:00<00:00, 3483.81it/s]
[transformers] RobertaModel LOAD REPORT from: guillermoruiz/mex_state
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 694
  0/694 tweets procesados

--- mex_large ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1191.48it/s]
[transformers] RobertaModel LOAD REPORT from: guillermoruiz/mex_large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 694
  0/694 tweets procesados

--- beto ---


...

Tweets a procesar: 1178
  0/1178 tweets procesados

--- bertin ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1188.17it/s]
[transformers] RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 647
  0/647 tweets procesados

--- robertuito ---


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8959.76it/s]
[transformers] RobertaModel LOAD REPORT from: pysentimiento/robertuito-base-cased
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 1127
  0/1127 tweets procesados
    modelo     palabra tipo_esperado  n_ocurrencias     config  min_cluster_size_usado  k_sentidos  pct_ruido
    bertin  aeropuerto    monosémica            138         10                      10           5  10.144928
    bertin  aeropuerto    monosémica            138          3                       3           9   5.072464
    bertin  aeropuerto    monosémica            138          5                       5           4   0.000000
    bertin  aeropuerto    monosémica            138 adaptativo                       7           3   3.623188
      beto  aeropuerto    monosémica            141         10                      10           2   0.000000
      beto  aeropuerto    monosémica            141          3                       3           3   0.000000
      beto  aeropuerto    monosémica            141          5                       5           3   0.000000
      beto  aeropuerto    monosémica            141 adaptativo       

In [10]:
# Obtener embeddings
for nombre_modelo, hf_id in MODELOS.items():
    embeddings_por_palabra, tweets_por_palabra = extraer_embeddings_modelo(
        nombre_modelo, 
        hf_id, 
        TOKENS_CSV[nombre_modelo], 
        PALABRAS_CSV[nombre_modelo], 
        #['banco', 'pobre', 'vida']
        palabras_objetivo
    )
    guardar_embeddings_modelo(nombre_modelo, embeddings_por_palabra, tweets_por_palabra)
    with open(f"embeddings/{nombre_modelo}_tweets_por_palabra.json", "w") as file:
        json.dump(tweets_por_palabra, file)
    


--- mex_state ---


Loading weights: 100%|██████████| 101/101 [00:00<00:00, 2971.68it/s]
[transformers] RobertaModel LOAD REPORT from: guillermoruiz/mex_state
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 76423
  0/76423 tweets procesados
  1600/76423 tweets procesados
  3200/76423 tweets procesados
  4800/76423 tweets procesados
  6400/76423 tweets procesados
  8000/76423 tweets procesados
  9600/76423 tweets procesados
  11200/76423 tweets procesados
  12800/76423 tweets procesados
  14400/76423 tweets procesados
  16000/76423 tweets procesados
  17600/76423 tweets procesados
  19200/76423 tweets procesados
  20800/76423 tweets procesados
  22400/76423 tweets procesados
  24000/76423 tweets procesados
  25600/76423 tweets procesados
  27200/76423 tweets procesados
  28800/76423 tweets procesados
  30400/76423 tweets procesados
  32000/76423 tweets procesados
  33600/76423 tweets procesados
  35200/76423 tweets procesados
  36800/76423 tweets procesados
  38400/76423 tweets procesados
  40000/76423 tweets procesados
  41600/76423 tweets procesados
  43200/76423 tweets procesados
  44800/76423 tweets procesados
  46400/76423 tweets procesados
  48000/76423 tweets proc

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3129.92it/s]
[transformers] RobertaModel LOAD REPORT from: guillermoruiz/mex_large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 76423
  0/76423 tweets procesados
  1600/76423 tweets procesados
  3200/76423 tweets procesados
  4800/76423 tweets procesados
  6400/76423 tweets procesados
  8000/76423 tweets procesados
  9600/76423 tweets procesados
  11200/76423 tweets procesados
  12800/76423 tweets procesados
  14400/76423 tweets procesados
  16000/76423 tweets procesados
  17600/76423 tweets procesados
  19200/76423 tweets procesados
  20800/76423 tweets procesados
  22400/76423 tweets procesados
  24000/76423 tweets procesados
  25600/76423 tweets procesados
  27200/76423 tweets procesados
  28800/76423 tweets procesados
  30400/76423 tweets procesados
  32000/76423 tweets procesados
  33600/76423 tweets procesados
  35200/76423 tweets procesados
  36800/76423 tweets procesados
  38400/76423 tweets procesados
  40000/76423 tweets procesados
  41600/76423 tweets procesados
  43200/76423 tweets procesados
  44800/76423 tweets procesados
  46400/76423 tweets procesados
  48000/76423 tweets proc

...

Tweets a procesar: 76354
  0/76354 tweets procesados
  1600/76354 tweets procesados
  3200/76354 tweets procesados
  4800/76354 tweets procesados
  6400/76354 tweets procesados
  8000/76354 tweets procesados
  9600/76354 tweets procesados
  11200/76354 tweets procesados
  12800/76354 tweets procesados
  14400/76354 tweets procesados
  16000/76354 tweets procesados
  17600/76354 tweets procesados
  19200/76354 tweets procesados
  20800/76354 tweets procesados
  22400/76354 tweets procesados
  24000/76354 tweets procesados
  25600/76354 tweets procesados
  27200/76354 tweets procesados
  28800/76354 tweets procesados
  30400/76354 tweets procesados
  32000/76354 tweets procesados
  33600/76354 tweets procesados
  35200/76354 tweets procesados
  36800/76354 tweets procesados
  38400/76354 tweets procesados
  40000/76354 tweets procesados
  41600/76354 tweets procesados
  43200/76354 tweets procesados
  44800/76354 tweets procesados
  46400/76354 tweets procesados
  48000/76354 tweets proc

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3126.86it/s]
[transformers] RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 76002
  0/76002 tweets procesados
  1600/76002 tweets procesados
  3200/76002 tweets procesados
  4800/76002 tweets procesados
  6400/76002 tweets procesados
  8000/76002 tweets procesados
  9600/76002 tweets procesados
  11200/76002 tweets procesados
  12800/76002 tweets procesados
  14400/76002 tweets procesados
  16000/76002 tweets procesados
  17600/76002 tweets procesados
  19200/76002 tweets procesados
  20800/76002 tweets procesados
  22400/76002 tweets procesados
  24000/76002 tweets procesados
  25600/76002 tweets procesados
  27200/76002 tweets procesados
  28800/76002 tweets procesados
  30400/76002 tweets procesados
  32000/76002 tweets procesados
  33600/76002 tweets procesados
  35200/76002 tweets procesados
  36800/76002 tweets procesados
  38400/76002 tweets procesados
  40000/76002 tweets procesados
  41600/76002 tweets procesados
  43200/76002 tweets procesados
  44800/76002 tweets procesados
  46400/76002 tweets procesados
  48000/76002 tweets proc

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9858.59it/s]
[transformers] RobertaModel LOAD REPORT from: pysentimiento/robertuito-base-cased
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tweets a procesar: 76234
  0/76234 tweets procesados
  1600/76234 tweets procesados
  3200/76234 tweets procesados
  4800/76234 tweets procesados
  6400/76234 tweets procesados
  8000/76234 tweets procesados
  9600/76234 tweets procesados
  11200/76234 tweets procesados
  12800/76234 tweets procesados
  14400/76234 tweets procesados
  16000/76234 tweets procesados
  17600/76234 tweets procesados
  19200/76234 tweets procesados
  20800/76234 tweets procesados
  22400/76234 tweets procesados
  24000/76234 tweets procesados
  25600/76234 tweets procesados
  27200/76234 tweets procesados
  28800/76234 tweets procesados
  30400/76234 tweets procesados
  32000/76234 tweets procesados
  33600/76234 tweets procesados
  35200/76234 tweets procesados
  36800/76234 tweets procesados
  38400/76234 tweets procesados
  40000/76234 tweets procesados
  41600/76234 tweets procesados
  43200/76234 tweets procesados
  44800/76234 tweets procesados
  46400/76234 tweets procesados
  48000/76234 tweets proc

# Hacer clusters

In [131]:
emb, w = cargar_embeddings('mex_state')
emb /= np.linalg.norm(emb, axis=1).reshape(-1, 1)

In [132]:
df = pd.DataFrame(list(w))
df['region_codes'] = pd.Categorical(df['region']).codes
emb.shape, df.shape

((382668, 512), (382668, 4))

In [ ]:
import csv
 
RESULTADOS_CSV = "resultados_clustering.csv"
CAMPOS_RESULTADOS = ["word_normalized", "modelo", "n_ocurrencias", "k_sentidos", "pct_ruido", "es_polisemica"]
UMBRAL_RUIDO_NO_DETERMINADO = 50  # % de ruido a partir del cual no se puede determinar polisemia
 
 
def cargar_procesados(ruta=RESULTADOS_CSV):
    """Devuelve el set de (modelo, word_normalized) ya procesados, leyendo el
    CSV si existe. Tolera una última línea corrupta (ej. si el proceso se cayó
    a mitad de una escritura)."""
    if not os.path.exists(ruta):
        return set()
    try:
        df_previo = pd.read_csv(ruta, on_bad_lines="skip")
    except TypeError:  # pandas viejo sin on_bad_lines
        df_previo = pd.read_csv(ruta, error_bad_lines=False)
    return set(zip(df_previo["modelo"], df_previo["word_normalized"]))
 
 
def clusterizar_todas_las_palabras_resumible(nombre_modelo, min_cluster_size=None, min_samples=None, ruta=RESULTADOS_CSV):
    """Igual que clusterizar_todas_las_palabras, pero guarda el resultado de
    CADA palabra apenas termina (append + flush a `ruta`) y se salta las
    palabras que ya estén en el archivo — así, si el proceso se cae a mitad
    (ej. por memoria de GPU), puedes volver a correr esta misma función y
    retoma donde se quedó, sin reprocesar lo ya hecho."""
    emb, w = cargar_embeddings(nombre_modelo)
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    palabras_arr = np.array([d["palabra"] for d in w])
    palabras_unicas = np.unique(palabras_arr)
 
    procesados = cargar_procesados(ruta)
    escribir_header = not os.path.exists(ruta)
    ya_hechas = sum(1 for p in palabras_unicas if (nombre_modelo, p) in procesados)
    print(f"--- {nombre_modelo}: {ya_hechas}/{len(palabras_unicas)} palabras ya procesadas, retomando ---")
 
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS_RESULTADOS)
        if escribir_header:
            writer.writeheader()
            f.flush()
 
        for i, palabra in enumerate(palabras_unicas):
            if (nombre_modelo, palabra) in procesados:
                continue
 
            idx = np.where(palabras_arr == palabra)[0]
            embs_palabra = emb[idx]
            n = len(embs_palabra)
 
            if n < MIN_OCURRENCIAS_PARA_CLUSTER:
                k, pct_ruido, es_polisemica = None, None, None
            else:
                labels = _hdbscan_labels(
                    embs_palabra,
                    min_cluster_size or HDBSCAN_MIN_CLUSTER_SIZE,
                    min_samples or HDBSCAN_MIN_SAMPLES,
                )
                k = len(set(labels) - {-1})
                pct_ruido = (labels == -1).mean() * 100
                es_polisemica = None if pct_ruido > UMBRAL_RUIDO_NO_DETERMINADO else (k > 1)
 
            writer.writerow({
                "word_normalized": palabra, "modelo": nombre_modelo,
                "n_ocurrencias": n, "k_sentidos": k, "pct_ruido": pct_ruido,
                "es_polisemica": es_polisemica,
            })
            f.flush()  # persiste en disco apenas termina esta palabra, no al final del loop
 
            if i % 500 == 0:
                print(f"  [{nombre_modelo}] {i}/{len(palabras_unicas)} palabras procesadas")
 
    print(f"--- {nombre_modelo}: completado ---")
 
 
for nombre_modelo in MODELOS:
    clusterizar_todas_las_palabras_resumible(nombre_modelo)
 

In [144]:
resultados_df = pd.read_csv(RESULTADOS_CSV)

print(f"\nTotal de filas (palabra x modelo): {len(resultados_df)}")
resultados_df.head(20) 



Total de filas (palabra x modelo): 62170


,word_normalized,modelo,n_ocurrencias,k_sentidos,pct_ruido,es_polisemica
0,aaaaah,mex_state,8,0.0,100.000000,NaN
1,aaaah,mex_state,7,NaN,NaN,NaN
2,aaaay,mex_state,6,NaN,NaN,NaN
3,aaah,mex_state,7,NaN,NaN,NaN
4,abajo,mex_state,98,7.0,28.571429,True
5,abandonada,mex_state,9,0.0,100.000000,NaN
6,abandonadas,mex_state,5,NaN,NaN,NaN
7,abandonado,mex_state,14,2.0,7.142857,True
8,abandonados,mex_state,8,0.0,100.000000,NaN
9,abandonar,mex_state,9,0.0,100.000000,NaN


# Visualizar tweets

In [133]:
data = pd.read_csv(PALABRAS_CSV['mex_state'])

In [134]:
idx = 21
get_tweet(data, w[idx]['tweet_id'], w[idx]['palabra'])

hidalgo _geo hoy no tengo nada que twittear que raro no 🤔 
